# FWI Gradient and Optimization

Derive the gradient, verify it numerically, and investigate an update.

## Computational conventions and data provenance
We solve the 2-D constant-density isotropic acoustic equation on a teaching grid. GeoFWI supplies the velocity values; the physical grid spacing and acquisition are assigned for this course, not field metadata. Numerical experiments share the same forward operator for observations and predictions (an inverse crime); they demonstrate principles rather than field-data performance. The sponge is approximate, and second-order spatial differences can be dispersive. Check points per shortest wavelength and reduce frequency or grid spacing for quantitative modeling.

**Data:** Li et al. (2026), *GeoFWI: A large velocity model dataset for benchmarking full waveform inversion using deep learning*, [paper](https://doi.org/10.1029/2025JH001037), [dataset](https://doi.org/10.5281/zenodo.18603709), [repository](https://github.com/aaspip/geofwi). The original archive contains 49,476 models of 100 × 100 samples. The loader requests only the selected sample where HTTP byte ranges are supported; otherwise it downloads the full archive. No synthetic substitute is silently used.

**Run in Colab:** Sign in, optionally select Runtime → Change runtime type → GPU, then execute cells in order. Each notebook is standalone. If download fails, retry or upload the official `geofwi.npy` into `geofwi_cache/`. A fresh runtime loses its local cache. Save a copy in Drive to retain edits.


In [ ]:
# Colab normally includes these packages; install only missing dependencies.
import importlib.util, subprocess, sys
for package in ['numpy','scipy','matplotlib','torch','requests']:
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable,'-m','pip','install',package])
import numpy as np, torch, matplotlib.pyplot as plt, requests, io, re, ast
from pathlib import Path
from scipy.ndimage import gaussian_filter, zoom
from IPython.display import display
torch.manual_seed(7); np.random.seed(7)
torch.set_num_threads(2)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
plt.rcParams.update({'figure.dpi':120,'axes.grid':False,'font.size':10})
print('Device:',device,'PyTorch:',torch.__version__)


In [ ]:
# Retrieve exact byte ranges from the official GeoFWI NPY archive.
# If the server ignores Range, optionally download the entire ~2 GB archive once.
URL='https://zenodo.org/records/18603709/files/geofwi.npy?download=1'
ALLOW_FULL_DOWNLOAD=True
cache=Path('geofwi_cache'); cache.mkdir(exist_ok=True)
full=cache/'geofwi.npy'
def fetch_range(a,b):
    with requests.get(URL,headers={'Range':f'bytes={a}-{b}','Accept-Encoding':'identity'},stream=True,timeout=(30,120)) as r:
        r.raise_for_status()
        if r.status_code==206:
            if not r.headers.get('Content-Range','').startswith(f'bytes {a}-{b}/'):
                raise ValueError('Unexpected range response')
            raw=r.content
            if len(raw)!=b-a+1: raise IOError('Incomplete range')
            return raw
        if not ALLOW_FULL_DOWNLOAD: raise RuntimeError('Range unavailable; enable full download or supply archive.')
        print('Range unsupported: downloading full GeoFWI archive (~2 GB).')
        tmp=full.with_suffix('.part')
        with tmp.open('wb') as f:
            for chunk in r.iter_content(1024*1024): f.write(chunk)
        arr=np.load(tmp,mmap_mode='r',allow_pickle=False)
        assert arr.shape==(49476,100,100)
        del arr; tmp.replace(full)
        with full.open('rb') as f: f.seek(a); return f.read(b-a+1)
def get_model(index):
    assert 0<=index<49476
    path=cache/f'model_{index}.npy'
    if path.exists(): return np.load(path,allow_pickle=False)
    if full.exists(): v=np.array(np.load(full,mmap_mode='r')[index],dtype=np.float32)
    else:
        header=fetch_range(0,4095)
        f=io.BytesIO(header); version=np.lib.format.read_magic(f)
        shape,fortran,dtype=np.lib.format._read_array_header(f,version)
        assert shape==(49476,100,100) and not fortran, (shape,fortran)
        count=100*100*dtype.itemsize; start=f.tell()+index*count
        raw=fetch_range(start,start+count-1)
        v=np.frombuffer(raw,dtype=dtype).reshape(100,100).astype('float32')
    assert np.isfinite(v).all() and v.min()>500 and v.max()<10000, 'Check velocity units'
    np.save(path,v); return v
MODEL_INDEX=100
# Published 100 x 100 array; use a smaller grid for classroom speed.
# We interpret rows as depth, columns as horizontal position.
v_native=get_model(MODEL_INDEX)
N=48; dx=20.; dt=0.0015; nt=300
v_true_np=zoom(v_native,(N/100,N/100),order=1).astype('float32')
v_true=torch.tensor(v_true_np,device=device)
assert float(v_true.max())*dt/dx<1/np.sqrt(2)
print('GeoFWI index:',MODEL_INDEX,'velocity range:',v_true_np.min(),v_true_np.max())


## Shared numerical engine
The following implementation is deliberately explicit so that every operation is inspectable. Source amplitude is in arbitrary units; the discrete forcing already includes the time-step scaling convention.

In [ ]:
# A differentiable second-order acoustic finite-difference solver.
# Array ordering: [shot, depth, horizontal]. All units are SI.
def ricker(f, t0=None):
    t=torch.arange(nt,device=device)*dt
    a=(torch.pi*f*(t-(1.5/f if t0 is None else t0)))**2
    return (1-2*a)*torch.exp(-a)
def forward(v, f=15., shots=(8,24,39), snapshots=False):
    assert float(v.detach().max())*dt/dx<1/np.sqrt(2), 'CFL violation'
    ns=len(shots); zsrc=5; zrec=5
    src=torch.zeros((ns,N,N),device=device)
    for i,x in enumerate(shots): src[i,zsrc,x]=1
    coords=torch.arange(N,device=device)
    # Sponge on all boundaries, not a free surface or PML.
    edge=torch.minimum(coords,N-1-coords).float()
    taper=torch.exp(-0.12*((5-edge).clamp(min=0)/5)**2)
    damp=taper[:,None]*taper[None,:]
    u=torch.zeros_like(src); old=torch.zeros_like(src)
    rec=[]; frames=[]; w=ricker(f); c=(v*dt/dx)**2
    for k in range(nt):
        p=torch.nn.functional.pad(u,(1,1,1,1))
        lap=p[:,2:,1:-1]+p[:,:-2,1:-1]+p[:,1:-1,2:]+p[:,1:-1,:-2]-4*u
        new=damp*(2*u-old+c*lap+src*w[k])
        old,u=u,new
        rec.append(u[:,zrec,5:-5])
        if snapshots and k in (60,120,180,240): frames.append(u[0].detach().cpu().numpy())
    result=torch.stack(rec,dim=1)
    return (result,frames) if snapshots else result
def arr(x): return x.detach().cpu().numpy() if torch.is_tensor(x) else np.asarray(x)
def modelplot(ax,v,title,limits=None):
    im=ax.imshow(arr(v),cmap='turbo',extent=[0,N*dx/1000,N*dx/1000,0],aspect='equal',**({} if limits is None else dict(vmin=limits[0],vmax=limits[1])))
    ax.set(xlabel='Distance (km)',ylabel='Depth (km)',title=title); plt.colorbar(im,ax=ax,label='m/s',shrink=.8)
def gather(ax,d,title,clip=None):
    a=arr(d); clip=np.percentile(np.abs(a),98) if clip is None else clip
    im=ax.imshow(a,aspect='auto',cmap='gray',vmin=-max(clip,1e-8),vmax=max(clip,1e-8),extent=[5*dx/1000,(N-6)*dx/1000,nt*dt,0]); ax.set(xlabel='Receiver x (km)',ylabel='Time (s)',title=title); plt.colorbar(im,ax=ax,shrink=.8,label='Amplitude (a.u.)')
def loss(pred,obs): return .5*(pred-obs).square().mean()/(obs.square().mean()+1e-12)
v0=torch.tensor(gaussian_filter(v_true_np,5),device=device)


## 1. Adjoint and chain rule
Write $A(v)p=s$, $r=Rp-d$. For $\Phi=\tfrac12 r^Tr$, solve $A^T\lambda=R^Tr$. Differentiating the wave equation gives
$$\delta\Phi=-\lambda^T(\delta A)p.$$
For $A=m\partial_t^2-\nabla^2$, $m=v^{-2}$ and model-independent forcing,
$$g_m=-\int\lambda p_{tt}\,dt,\qquad g_v=-2v^{-3}g_m.$$
The exact discrete gradient depends on the discretization and source convention. Our solver uses a velocity-independent forcing in the acceleration update; PyTorch differentiates that actual discrete recurrence, rather than assuming the above continuous source convention. Reverse-mode differentiation is the discrete adjoint of the implemented computational graph.


In [ ]:
with torch.no_grad(): obs=forward(v_true)
q=(v0/2000).detach().requires_grad_(); J=loss(forward(2000*q),obs); J.backward(); g=q.grad.detach()
fig,ax=plt.subplots(1,2,figsize=(10,4)); modelplot(ax[0],v0,'Initial model'); lim=np.percentile(np.abs(arr(g)),99); im=ax[1].imshow(arr(g),cmap='seismic',vmin=-lim,vmax=lim); ax[1].set(title='Gradient w.r.t. dimensionless velocity',xlabel='x sample',ylabel='z sample'); fig.colorbar(im,ax=ax[1]); plt.tight_layout(); plt.show()

## 2. Taylor test
For a unit perturbation $h$, the linear prediction is $\Phi(q+\epsilon h)=\Phi(q)+\epsilon g^Th+O(\epsilon^2)$. The remainder should decrease quadratically until roundoff dominates. Use the normalized gradient direction to avoid a vanishing directional derivative.

In [ ]:
h=g/g.norm(); eps=np.logspace(-3,-.5,8); e0=[]; e1=[]
with torch.no_grad():
 for e in eps:
  diff=float(loss(forward(2000*(q+e*h)),obs)-J); e0.append(abs(diff)); e1.append(abs(diff-e*float((g*h).sum())))
plt.loglog(eps,e0,'o-',label='Uncorrected'); plt.loglog(eps,e1,'s-',label='Taylor remainder'); plt.loglog(eps,e0[-1]*eps/eps[-1],'--',label='Slope 1'); plt.loglog(eps,max(e1[-1],1e-12)*(eps/eps[-1])**2,':',label='Slope 2'); plt.xlabel('Perturbation'); plt.ylabel('Absolute error'); plt.legend(); plt.show()

## 3. Gradient descent and line search
A descent step is $q_{k+1}=q_k-\alpha g_k$. Scan several candidate step lengths. Too large a step may increase the objective; normalization makes the trial step interpretable.

In [ ]:
alphas=np.linspace(0,2,15); values=[]
with torch.no_grad():
 for a in alphas: values.append(float(loss(forward(2000*(q-a*h).clamp(.65,2.25)),obs)))
plt.plot(alphas,values,'o-'); plt.xlabel('Step length along normalized gradient'); plt.ylabel('Objective'); plt.show()
print('Best trial:',alphas[np.argmin(values)])

## 4. Exercises
Compare gradient illumination for one and three shots. Try a random Taylor direction. Explain why an adjoint test is not proof of physical accuracy.